# Notebook 2 — Create the labels

**Job of this notebook:** build the `is_late` label from the actual
delivery date vs. the estimated delivery date, sanity-check it on real
orders, and look at the class balance.

**Reads:** `data/interim/ml_table.parquet` (Notebook 1's artifact).
**Writes:** `data/interim/labeled_table.parquet`.


In [ ]:
import sys
sys.path.append("../src")

import pandas as pd
from config import ML_TABLE_PATH, LABELED_TABLE_PATH, LABEL_COL

ml_table = pd.read_parquet(ML_TABLE_PATH)
ml_table.shape


## 1. Only orders that were actually delivered can be labeled

Orders that are cancelled, still processing, or otherwise never reached
`order_delivered_customer_date` have no delivery date to compare against
the estimate. We can't label those — drop them and say so explicitly,
rather than silently letting them turn into NaN labels later.


In [ ]:
date_cols = [
    "order_purchase_timestamp", "order_approved_at",
    "order_delivered_carrier_date", "order_delivered_customer_date",
    "order_estimated_delivery_date",
]
for c in date_cols:
    ml_table[c] = pd.to_datetime(ml_table[c])

print(ml_table["order_status"].value_counts())
print()
print("Missing order_delivered_customer_date:", ml_table["order_delivered_customer_date"].isna().sum())

labeled = ml_table[ml_table["order_delivered_customer_date"].notna()].copy()
print(f"Kept {len(labeled)} / {len(ml_table)} orders that have an actual delivery date")


## 2. Build the label

`is_late = 1` when the order was delivered **after** the estimated
delivery date, else `0`.


In [ ]:
labeled["delivery_delay_days"] = (
    labeled["order_delivered_customer_date"] - labeled["order_estimated_delivery_date"]
).dt.total_seconds() / 86400

labeled["is_late"] = (labeled["delivery_delay_days"] > 0).astype(int)

labeled[["order_id", "order_estimated_delivery_date", "order_delivered_customer_date",
         "delivery_delay_days", "is_late"]].head(10)


## 3. Sanity-check the label against real orders

Eyeball a few clearly-on-time and clearly-late orders to make sure the
sign of `delivery_delay_days` and the label agree with what you'd expect
by just reading the two dates.


In [ ]:
cols = ["order_id", "order_estimated_delivery_date", "order_delivered_customer_date",
        "delivery_delay_days", "is_late"]

print("Clearly late examples:")
display(labeled.sort_values("delivery_delay_days", ascending=False)[cols].head(3))

print("Clearly on-time examples:")
display(labeled.sort_values("delivery_delay_days")[cols].head(3))

# A delivery exactly on the estimated date is on-time, not late
edge_cases = labeled[labeled["delivery_delay_days"] == 0]
print(f"Exact-day deliveries: {len(edge_cases)} — labeled on-time by the '> 0' rule above")


## 4. Class distribution — is this imbalanced?

In [ ]:
counts = labeled["is_late"].value_counts()
pct = labeled["is_late"].value_counts(normalize=True) * 100

print("Counts:")
print(counts)
print()
print("Percent:")
print(pct.round(2))

import matplotlib.pyplot as plt
labeled["is_late"].value_counts().plot(kind="bar", title="Late (1) vs On-time (0)")
plt.xlabel("is_late")
plt.ylabel("orders")
plt.show()


**Imbalance note:** with typical Olist data the late-delivery class sits
around 6–8% of orders — a real minority-class imbalance problem. Write
down the actual number you got above; it decides:
- the split strategy in Notebook 3 (stratify to keep the ratio stable),
- the metric choice in Notebook 6 (accuracy would be misleading here).


## Artifact: `labeled_table.parquet`

In [ ]:
labeled.to_parquet(LABELED_TABLE_PATH, index=False)
print(f"Saved {labeled.shape} to {LABELED_TABLE_PATH}")
